In [ ]:
import ee
import geemap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from IPython.display import display, HTML

ee.Authenticate()
ee.Initialize(project='gee-space-hack') #Reemplazar el nombre por el proyecto que se asignara al momento de crear la conexion entre google cloud y GEE

# ======================== CONFIGURACION ========================

AOI = ee.FeatureCollection('projects/gee-space-hack/assets/SpaceHack/AOI_GreaterGuayaquil_v2')
ASSET_ROOT = 'projects/gee-space-hack/assets/SpaceHack'

In [ ]:
print('=' * 60)
print('PARTE 1: CONTEXTO SOCIOECONOMICO Y POLITICO')
print('=' * 60)

context = """
GREATER GUAYAQUIL - PERFIL DE VULNERABILIDAD
=============================================

DATOS DEMOGRAFICOS:
  Poblacion total: ~3.3 millones de habitantes
  Guayaquil: 2.7M | Duran: 315K | Samborondon: 102K | Daule: 160K
  Densidad: alta concentracion en margenes del rio Guayas
  Asentamientos informales en zonas inundables: ~30% de la poblacion urbana

VULNERABILIDAD CLIMATICA:
  Lluvias extremas: >70mm en un solo dia (registros 2023-2025)
  Mareas del rio Guayas: hasta 5 metros
  Evento compuesto: lluvia + marea alta = colapso del drenaje urbano
  Fenomeno de El Nino: incrementa frecuencia de eventos extremos
  Proyeccion: aumento del nivel del mar de 0.3-0.6m para 2050 (IPCC AR6)

IMPACTO ECONOMICO DE INUNDACIONES:
  Guayaquil concentra ~25% del PIB de Ecuador
  Puerto de Guayaquil: principal puerto del pais
  Perdidas por inundacion 2023: estimadas en >$50M USD (infraestructura)
  Costo de dano promedio: $50,000 USD/ha urbana inundada (World Bank)

ECOSISTEMA DE MANGLAR:
  Golfo de Guayaquil: mayor estuario del Pacifico sudamericano
  Especies principales: Rhizophora mangle, Avicennia germinans, Laguncularia racemosa
  Servicios ecosistemicos: proteccion costera, captura de carbono, habitat pesquero
  Produccion camaronera: principal amenaza historica al manglar
  Tasa de perdida 1970-2000: ~50% del manglar original
"""
print(context)

stakeholders = """
STAKEHOLDERS Y MARCO POLITICO
==============================

ACTORES CLAVE:
  1. GAD Municipal de Guayaquil - planificacion urbana y drenaje
  2. Ministerio del Ambiente (MAATE) - custodio legal de los manglares
  3. SENAGUA - gestion de recursos hidricos y riesgo de inundacion
  4. Secretaria de Gestion de Riesgos (SGR) - alertas tempranas
  5. Comunidades ancestrales de manglar - usuarios de custodias
  6. Sector camaronero - principal driver de deforestacion de manglar
  7. INOCAR - datos de mareas y nivel del mar

MARCO LEGAL:
  - Codigo Organico del Ambiente (2017): Art. 99-103 sobre manglares
  - Acuerdo Ministerial 129: regula concesiones en zonas de manglar
  - Plan Nacional de Manejo de Manglares (MAATE)
  - Custodias de manglar: figura legal que otorga gestion comunitaria
  - Ecuador signatario del Acuerdo de Paris y Convenio de Ramsar
  - Convencion de Cartagena sobre contaminacion marina

POLITICAS RELEVANTES:
  - Plan de Uso y Gestion del Suelo de Guayaquil (PUGS 2022)
  - Plan de Adaptacion al Cambio Climatico de Ecuador (2023)
  - Estrategia Nacional de Biodiversidad 2030
  - ODS 11 (ciudades sostenibles), ODS 13 (accion climatica), ODS 14 (vida submarina)
"""
print(stakeholders)


In [ ]:
# ======================== PARTE 2: RESULTADOS TECNICOS ========================

print('=' * 60)
print('PARTE 2: RESULTADOS TECNICOS')
print('=' * 60)

# --- 2.1 Cargar manglares SERVIR ---
print('\n--- 2.1 Cobertura de manglar historica (SERVIR-Amazonia) ---')

man_years = {
    2018: ee.FeatureCollection(f'{ASSET_ROOT}/data/MAN_2018'),
    2020: ee.FeatureCollection(f'{ASSET_ROOT}/data/MAN_2020'),
    2022: ee.FeatureCollection(f'{ASSET_ROOT}/data/MAN_2022'),
}

mangrove_rasters = {}
for year, fc in man_years.items():
    mangrove_rasters[year] = (ee.Image(0)
        .paint(fc, 1)
        .rename('manglar')
        .clip(AOI.geometry()))

# Areas SERVIR
print('\nAreas de manglar (SERVIR-Amazonia):')
servir_areas = {}
for year, raster in mangrove_rasters.items():
    area = (raster.multiply(ee.Image.pixelArea()).divide(10000)
        .reduceRegion(ee.Reducer.sum(), AOI.geometry(), 10, maxPixels=1e10)
        .get('manglar').getInfo())
    servir_areas[year] = area
    print(f'  {year}: {area:,.1f} ha')

In [ ]:
# --- 2.2 Clasificacion propia (si assets disponibles) ---
print('\n--- 2.2 Clasificacion Random Forest 2023-2025 ---')

classified_years = {}
rf_areas = {}

for year in [2023, 2024, 2025]:
    try:
        img = ee.Image(f'{ASSET_ROOT}/MANGROVE_{year}')
        classified_years[year] = img
        area = (img.multiply(ee.Image.pixelArea()).divide(10000)
            .reduceRegion(ee.Reducer.sum(), AOI.geometry(), 10, maxPixels=1e10)
            .get('manglar').getInfo())
        rf_areas[year] = area
        print(f'  {year}: {area:,.1f} ha')
    except Exception as e:
        print(f'  {year}: asset no disponible aun ({e})')
        print(f'         Usar datos de M2 cuando los exports terminen')

# Combinar todas las areas
all_areas = {**servir_areas, **rf_areas}


In [ ]:
# ======================== PARTE 3: MODELO DE INUNDACION ========================

print('\n' + '=' * 60)
print('PARTE 3: MODELO DE INUNDACION')
print('=' * 60)

# DEM
dem = ee.Image('USGS/SRTMGL1_003').select('elevation').clip(AOI.geometry())

# Manglar actual (usar el mas reciente disponible)
latest_year = max(mangrove_rasters.keys())
mangrove_current = mangrove_rasters[latest_year]
print(f'\nUsando manglar de referencia: {latest_year}')

# Poblacion
worldpop = (ee.ImageCollection('WorldPop/GP/100m/pop/ECU')
    .filterDate('2020-01-01', '2020-12-31')
    .mosaic().clip(AOI.geometry()).rename('population'))

# Urbano
ghsl = ee.Image('JRC/GHSL/P2023A/GHS_BUILT_S/2020').clip(AOI.geometry())
urban_mask = ghsl.select('built_surface').gt(0).rename('urban')

# Ancho de manglar
mangrove_width = (mangrove_current
    .fastDistanceTransform(256).sqrt().multiply(10)
    .rename('mangrove_width'))

# Atenuacion
ATTENUATION_CM_PER_100M = 30
MAX_ATTENUATION_M = 1.5
attenuation = (mangrove_width
    .divide(100).multiply(ATTENUATION_CM_PER_100M / 100)
    .min(MAX_ATTENUATION_M).rename('attenuation'))

# Escenarios
FLOOD_SCENARIOS = {
    'Marea normal (3m)': 3.0,
    'Marea alta (4m)': 4.0,
    'Extremo: marea+lluvia (5m)': 5.0,
    'Catastrofico (6m)': 6.0
}

COST_PER_HA_USD = 50000

results = []

for name, level in FLOOD_SCENARIOS.items():
    # CON manglar
    effective = ee.Image.constant(level).subtract(attenuation)
    flooded_with = dem.lt(effective).And(dem.lt(level)).rename('flooded')

    # SIN manglar
    flooded_without = dem.lt(level).rename('flooded')

    # Areas
    area_with = (flooded_with.multiply(ee.Image.pixelArea()).divide(10000)
        .reduceRegion(ee.Reducer.sum(), AOI.geometry(), 30, maxPixels=1e10)
        .get('flooded').getInfo())

    area_without = (flooded_without.multiply(ee.Image.pixelArea()).divide(10000)
        .reduceRegion(ee.Reducer.sum(), AOI.geometry(), 30, maxPixels=1e10)
        .get('flooded').getInfo())

    # Poblacion
    pop_with = (flooded_with.multiply(worldpop)
        .reduceRegion(ee.Reducer.sum(), AOI.geometry(), 100, maxPixels=1e10)
        .get('population').getInfo())

    pop_without = (flooded_without.multiply(worldpop)
        .reduceRegion(ee.Reducer.sum(), AOI.geometry(), 100, maxPixels=1e10)
        .get('population').getInfo())

    # Urbano inundado
    urban_with = (flooded_with.And(urban_mask)
        .multiply(ee.Image.pixelArea()).divide(10000)
        .reduceRegion(ee.Reducer.sum(), AOI.geometry(), 30, maxPixels=1e10)
        .values().get(0).getInfo())

    urban_without = (flooded_without.And(urban_mask)
        .multiply(ee.Image.pixelArea()).divide(10000)
        .reduceRegion(ee.Reducer.sum(), AOI.geometry(), 30, maxPixels=1e10)
        .values().get(0).getInfo())

    area_protected = (area_without or 0) - (area_with or 0)
    pop_protected = int(pop_without or 0) - int(pop_with or 0)
    urban_protected = (urban_without or 0) - (urban_with or 0)
    cost_avoided = urban_protected * COST_PER_HA_USD

    results.append({
        'Escenario': name,
        'Nivel (m)': level,
        'Area inundada CON manglar (ha)': round(area_with or 0, 1),
        'Area inundada SIN manglar (ha)': round(area_without or 0, 1),
        'Area protegida por manglar (ha)': round(area_protected, 1),
        'Poblacion expuesta CON manglar': int(pop_with or 0),
        'Poblacion expuesta SIN manglar': int(pop_without or 0),
        'Poblacion protegida': pop_protected,
        'Area urbana protegida (ha)': round(urban_protected, 1),
        'Danos evitados (USD)': round(cost_avoided, 0)
    })

    print(f'\n{name}:')
    print(f'  Area protegida: {area_protected:,.1f} ha')
    print(f'  Poblacion protegida: {pop_protected:,}')
    print(f'  Danos evitados: ${cost_avoided:,.0f} USD')

df_flood = pd.DataFrame(results)
print('\n--- Tabla resumen de inundacion ---')
display(df_flood)

In [ ]:
# ======================== PARTE 4: RESTAURACION ========================

print('\n' + '=' * 60)
print('PARTE 4: ESCENARIOS DE RESTAURACION')
print('=' * 60)

man2018_raster = (ee.Image(0)
    .paint(ee.FeatureCollection(f'{ASSET_ROOT}/data/MAN_2018'), 1)
    .rename('manglar').clip(AOI.geometry()))

# Manglar perdido 2018-2022
lost_mangrove = man2018_raster.And(mangrove_current.Not())

# Zonas restaurables (no urbanas, elevacion < 10m)
restorable = lost_mangrove.And(urban_mask.Not()).And(dem.lt(10)).rename('restorable')

area_restorable = (restorable.multiply(ee.Image.pixelArea()).divide(10000)
    .reduceRegion(ee.Reducer.sum(), AOI.geometry(), 10, maxPixels=1e10)
    .get('restorable').getInfo())
print(f'\nArea restaurable: {area_restorable:,.1f} ha')

# Manglar restaurado
mangrove_restored = mangrove_current.Or(restorable).rename('manglar')

# Recalcular atenuacion con restauracion
width_restored = mangrove_restored.fastDistanceTransform(256).sqrt().multiply(10)
atten_restored = width_restored.divide(100).multiply(ATTENUATION_CM_PER_100M / 100).min(MAX_ATTENUATION_M)

# Inundacion con restauracion (escenario extremo 5m)
level_extreme = 5.0
effective_restored = ee.Image.constant(level_extreme).subtract(atten_restored)
flooded_restored = dem.lt(effective_restored).And(dem.lt(level_extreme))

area_restored = (flooded_restored.multiply(ee.Image.pixelArea()).divide(10000)
    .reduceRegion(ee.Reducer.sum(), AOI.geometry(), 30, maxPixels=1e10)
    .get('manglar').getInfo())

pop_restored = (flooded_restored.multiply(worldpop)
    .reduceRegion(ee.Reducer.sum(), AOI.geometry(), 100, maxPixels=1e10)
    .get('population').getInfo())

# Comparar con escenario extremo actual (con manglar, sin restauracion)
current_extreme = next(r for r in results if r['Nivel (m)'] == 5.0)

print(f'\nEscenario extremo (5m) - Comparacion:')
print(f'  {"Metrica":<30} {"Actual":<20} {"Con restauracion":<20} {"Mejora":<20}')
print(f'  {"Area inundada (ha)":<30} {current_extreme["Area inundada CON manglar (ha)"]:<20,.1f} {area_restored or 0:<20,.1f} {(current_extreme["Area inundada CON manglar (ha)"] - (area_restored or 0)):<20,.1f}')
print(f'  {"Poblacion expuesta":<30} {current_extreme["Poblacion expuesta CON manglar"]:<20,} {int(pop_restored or 0):<20,} {current_extreme["Poblacion expuesta CON manglar"] - int(pop_restored or 0):<20,}')